In [10]:
import torch
import torch.nn as nn

import numpy as np
import math

from torch.nn.utils.prune import remove, l1_unstructured
from tqdm import tqdm
import torchsummary

from import_shelf import shelf
from shelf.dataloaders.cifar import get_CIFAR10_dataset
from shelf.trainers.classic import train, validate
from shelf.trainers.zeroth_order import gradient_fo
from shelf.pruners.scoring import get_grasp_score_dict, get_zo_grasp_score
from shelf.models.mlp_mixer import MLPMixer
from shelf.models.resnet.resnet9_cifar import ResNet9
from shelf.models.resnet.etc import resnet20


EPOCHS = 600
DEVICE = 'cuda'
PRUNING_RATE = 0.9
SUBDIM_RATE = 0.9
MASK_UPDATE_EPOCH = 20

train_loader, val_loader = get_CIFAR10_dataset(root='../data', batch_size=256)

model = resnet20().to(DEVICE)
torchsummary.summary(model, (3, 32, 32))

criterion = nn.CrossEntropyLoss().to(DEVICE)

pnames, params = zip(*model.named_parameters())
pmasks = {}
for mname, module in model.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        pmasks[mname + '.weight'] = (torch.rand_like(module.weight) > PRUNING_RATE).float().to(DEVICE)
for pname, param in zip(pnames, params):
    if pname not in pmasks:
        pmasks[pname] = torch.ones_like(param).to(DEVICE)

for pname, param in zip(pnames, params):
    num_total = param.numel()
    num_alive = pmasks[pname].sum().item()
    print(f'{pname}: {num_alive}/{num_total} = {num_alive/num_total:.2f}')

Files already downloaded and verified
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             432
       BatchNorm2d-2           [-1, 16, 32, 32]              32
            Conv2d-3           [-1, 16, 32, 32]           2,304
       BatchNorm2d-4           [-1, 16, 32, 32]              32
            Conv2d-5           [-1, 16, 32, 32]           2,304
       BatchNorm2d-6           [-1, 16, 32, 32]              32
        BasicBlock-7           [-1, 16, 32, 32]               0
            Conv2d-8           [-1, 16, 32, 32]           2,304
       BatchNorm2d-9           [-1, 16, 32, 32]              32
           Conv2d-10           [-1, 16, 32, 32]           2,304
      BatchNorm2d-11           [-1, 16, 32, 32]              32
       BasicBlock-12           [-1, 16, 32, 32]               0
           Conv2d-13           [-1, 16, 32, 32]           2,304
 

In [11]:
calc_r_by_gnum_cache = {}

def calc_r_by_gnum(N, d):
    global calc_r_by_gnum_cache

    if N not in calc_r_by_gnum_cache:
        calc_r_by_gnum_cache[N] = {}

    if d in calc_r_by_gnum_cache[N]:
        return calc_r_by_gnum_cache[N][d]

    equation = np.poly1d([1] + [0 for _ in range(N-1)] + [-d, d-1], False)
    roots = np.roots(equation)
    roots = roots[np.isreal(roots)]
    r = np.real(np.max(roots))

    calc_r_by_gnum_cache[N][d] = r

    if r <= 1:
        raise ValueError(f"r must be greater than 1: N={N}, d={d}, r={r}")

    return r

def create_group(param, pmask, score, num_group):
    num_alive = int(pmask.sum().item())
    r = calc_r_by_gnum(num_group, num_alive)

    score_selected = score.clone()[pmask == 1]
    score_sorted = torch.sort(score_selected.view(-1), descending=True).values

    group_index = torch.zeros_like(pmask)

    group_end_idx = 1
    group_size = 1
    milestones = []

    for group_idx in range(1, num_group+1):
        now_end_idx = math.floor(group_end_idx)
        if group_idx == num_group:
            now_end_idx = num_alive

        milestones.append(score_sorted[now_end_idx-1].item())
        
        group_size *= r
        group_start_idx = group_end_idx
        group_end_idx = group_start_idx + group_size
        if group_end_idx > num_alive:
            group_end_idx = num_alive

    for group_idx in range(num_group, 0, -1):
        mask = (score >= milestones[group_idx-1]) * pmask
        group_index[mask == 1] = group_idx
    
    return group_index


In [12]:
temp_mask = (torch.rand(100) > 0.5).float()
temp_score = torch.rand(100)
temp_group = create_group(params[0], temp_mask, temp_score, 10)

print(temp_mask)
print(temp_score)
print(temp_group)

for i in range(11):
    print(f"{i}: {torch.sum(temp_group == i)}, mean: {torch.mean(temp_score[temp_group == i])}")

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 1., 0., 1., 1., 1.,
        0., 0., 0., 0., 0., 1., 1., 0., 0., 1., 1., 0., 1., 1., 1., 0., 0., 0.,
        0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 0.,
        0., 1., 0., 1., 1., 1., 1., 0., 0., 1., 0., 1., 1., 1., 1., 0., 1., 0.,
        0., 1., 0., 0., 0., 0., 1., 0., 1., 1., 1., 1., 0., 1., 1., 1., 1., 0.,
        1., 0., 1., 0., 0., 0., 0., 0., 1., 1.])
tensor([0.9882, 0.3307, 0.3339, 0.3187, 0.1721, 0.6795, 0.2147, 0.1290, 0.7036,
        0.1996, 0.1782, 0.5912, 0.0510, 0.9184, 0.6221, 0.1624, 0.6483, 0.5881,
        0.0191, 0.5135, 0.7917, 0.6026, 0.6576, 0.2564, 0.5931, 0.8147, 0.1043,
        0.2996, 0.5437, 0.0185, 0.1842, 0.8955, 0.7833, 0.7138, 0.2408, 0.2911,
        0.4316, 0.5439, 0.3083, 0.7394, 0.6581, 0.6083, 0.2122, 0.6289, 0.2743,
        0.9413, 0.7531, 0.2842, 0.1810, 0.3877, 0.2827, 0.8990, 0.9706, 0.1700,
        0.0552, 0.0420, 0.2867, 0.0369, 0.1942, 0.8770, 0.4663, 0.0363,

In [24]:
x_temp, t_temp = next(iter(train_loader))
x_temp, t_temp = x_temp.to(DEVICE), t_temp.to(DEVICE)

def get_masked_gge(input, label, model, criterion, pmasks, subdim_rate):
    num_forward = 0

    real_gradient = gradient_fo(input, label, model, criterion)

    estimated_gradient = {pname: torch.zeros_like(param) for pname, param in zip(pnames, params)}

    for pname, param in zip(pnames, params):
        num_params = param.numel()
        num_alive = pmasks[pname].sum().item()

        num_query = math.ceil(num_alive * (1 - subdim_rate))
        num_group = max(int(math.sqrt(num_query)), 2)
        num_iter = max(num_query // num_group, 1)

        real_grad = real_gradient[pname]
        iter_grad = torch.randn_like(param)
        iter_group = create_group(param, pmasks[pname], iter_grad, num_group)

        for _ in range(num_iter):
            pnoise = torch.randn_like(param)
            iter_grad = torch.zeros_like(param)

            for group_idx in range(1, num_group+1):
                group_mask = (iter_group == group_idx).float()
                group_size = group_mask.sum().item()
                
                jvp_value = (real_grad * pnoise * group_mask).sum()
                iter_grad += jvp_value * pnoise * group_mask / (group_size ** 2)

                num_forward += 1
            
            estimated_gradient[pname] = estimated_gradient[pname] * 0.9 + iter_grad * 0.1
            iter_group = create_group(param, pmasks[pname], real_grad.abs(), num_group)
    
    print(f"Used {num_forward} forward passes")

    return estimated_gradient

def get_masked_rge(input, label, model, criterion, pmasks, subdim_rate):
    num_params = sum([param.numel() for param in model.parameters()])
    num_alive = sum([pmask.sum().item() for pmask in pmasks.values()])
    num_query = math.floor(num_alive * (1 - subdim_rate))
    pmasks_flat = torch.cat([pmasks[pname].view(-1) for pname in pnames])

    real_gradient = gradient_fo(input, label, model, criterion)
    real_gradient_flat = torch.cat([g.view(-1) for g in real_gradient.values()])

    estimated_gradient = {pname: torch.zeros_like(param) for pname, param in zip(pnames, params)}

    for q in tqdm(range(num_query)):
        pnoise = {pname: torch.randn_like(param) for pname, param in zip(pnames, params)}
        pnoise_flat = torch.cat([pnoise[pname].view(-1) for pname in pnames])

        jvp_value = (pnoise_flat * real_gradient_flat * pmasks_flat).sum()

        for pname, param in zip(pnames, params):
            estimated_gradient[pname] += jvp_value * pnoise[pname] * pmasks[pname]

    print(f"Used {num_query} forward passes")

    return estimated_gradient


real_gradient = gradient_fo(x_temp, t_temp, model, criterion)
estimated_gradient = get_masked_gge(x_temp, t_temp, model, criterion, pmasks, SUBDIM_RATE)
estimated_rge = get_masked_rge(x_temp, t_temp, model, criterion, pmasks, SUBDIM_RATE)

real_flat = torch.cat([g.view(-1) for g in real_gradient.values()])
estimated_gge_flat = torch.cat([g.view(-1) for g in estimated_gradient.values()])
estimated_rge_flat = torch.cat([g.view(-1) for g in estimated_rge.values()])
cossim_gge = torch.dot(real_flat, estimated_gge_flat) / (torch.norm(real_flat) * torch.norm(estimated_gge_flat))
cossim_rge = torch.dot(real_flat, estimated_rge_flat) / (torch.norm(real_flat) * torch.norm(estimated_rge_flat))

print(f"CosSim (GGE): {cossim_gge.item()}")
print(f"CosSim (RGE): {cossim_rge.item()}")
        

Used 2770 forward passes


100%|██████████| 2806/2806 [00:07<00:00, 351.04it/s]

Used 2806 forward passes
CosSim (GGE): 0.08748739957809448
CosSim (RGE): 0.0931934118270874
